- current age
- male
- years_to_retirement

In [1]:
import numpy as np
import pandas as pd

import statsmodels.api as sm
from scipy.stats import ks_2samp, mannwhitneyu
from sklearn.metrics import average_precision_score

from tqdm.auto import tqdm

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score, 
    precision_recall_curve,
    classification_report
)
pd.set_option("display.max_rows", None)
from statsmodels.tools.sm_exceptions import PerfectSeparationError

TARGET = "fraud"

%matplotlib inline

In [2]:
df = pd.read_parquet("../5DATA/dataset/train_stage2")

In [3]:
BASELINE = [
    # 거래 강도
    "log_abs_amount",
    "amount_income_ratio",
    "amount_limit_ratio",

    # 에러
    "has_error",

    # 고객 맥락 최소
    "credit_limit",
    "current_age",

    # 카드 특성
    "is_credit",
    "has_chip",
]

**current_age**

In [4]:
col = "current_age"

s = pd.to_numeric(df[col], errors="coerce")

pd.DataFrame({
    "count": [s.notna().sum()],
    "missing_rate": [s.isna().mean()],
    "mean": [s.mean()],
    "std": [s.std()],
    "skew": [s.skew()],
    "min": [s.min()],
    "p01": [s.quantile(0.01)],
    "p50": [s.quantile(0.50)],
    "p99": [s.quantile(0.99)],
    "max": [s.max()],
})

,count,missing_rate,mean,std,skew,min,p01,p50,p99,max
0,608430,0.0,54.572924,15.481459,0.687691,23,29.0,52.0,98.0,101


In [5]:
col = "current_age"

tmp = df[[col, "fraud"]].dropna().copy()
base = tmp["fraud"].mean()

tmp["age_bin"] = pd.qcut(tmp[col], 10, duplicates="drop")

g = tmp.groupby("age_bin")["fraud"]

age_lift = pd.DataFrame({
    "count": g.size(),
    "fraud_rate": g.mean(),
})

age_lift["lift"] = age_lift["fraud_rate"] / base
age_lift

/tmp/ipykernel_3861508/4047355867.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  g = tmp.groupby("age_bin")["fraud"]


,count,fraud_rate,lift
age_bin,,,
"(22.999, 36.0]",66232,0.009905,0.917516
"(36.0, 42.0]",69829,0.009967,0.923317
"(42.0, 46.0]",50374,0.009668,0.895571
"(46.0, 49.0]",71952,0.009562,0.885774
"(49.0, 52.0]",65568,0.008846,0.819433
"(52.0, 56.0]",53692,0.009797,0.907515
"(56.0, 60.0]",52807,0.011173,1.034994
"(60.0, 67.0]",67337,0.014643,1.356440
"(67.0, 80.0]",52949,0.014391,1.333137


In [6]:
LABEL = "fraud"

def fit_logit_and_score(df, features, y_col=LABEL):
    use_cols = features + [y_col]
    tmp = df[use_cols].dropna(subset=use_cols).copy()

    X = sm.add_constant(tmp[features], has_constant="add")
    y = tmp[y_col].astype(int)

    X = X.apply(pd.to_numeric, errors="raise")

    model = sm.Logit(y, X).fit(disp=0)
    score = model.predict(X)
    return model, y, score


def top_decile_lift(y_true, score, q=0.90):
    y_true = pd.Series(y_true).astype(int)
    score = pd.Series(score)

    base_rate = y_true.mean()
    thr = score.quantile(q)

    top_mask = score >= thr
    top_rate = y_true[top_mask].mean()
    lift = top_rate / base_rate if base_rate > 0 else np.nan

    return {
        "base_rate": float(base_rate),
        "top_decile_rate": float(top_rate),
        "top_decile_lift": float(lift),
        "thr": float(thr),
        "top_n": int(top_mask.sum()),
    }

rows_uni = []

model, y, score = fit_logit_and_score(df, [col], y_col=LABEL)

coef = model.params.get(col, np.nan)
pval = model.pvalues.get(col, np.nan)
or_val = float(np.exp(coef)) if pd.notnull(coef) else np.nan

lift = top_decile_lift(y, score)

rows_uni.append({
        "feature": col,
        "coef": float(coef) if pd.notnull(coef) else np.nan,
        "OR": or_val,
        "p_value": float(pval) if pd.notnull(pval) else np.nan,
        "top_decile_rate": lift["top_decile_rate"],
        "top_decile_lift": lift["top_decile_lift"],
        "top_n": lift["top_n"],
    })

uni_df = (
    pd.DataFrame(rows_uni)
      .sort_values("top_decile_lift", ascending=False)
      .reset_index(drop=True)
)

display(uni_df)

,feature,coef,OR,p_value,top_decile_rate,top_decile_lift,top_n
0,current_age,0.005249,1.005263,1.740944e-11,0.009717,0.900147,64011


> drop

**male**

In [7]:
col = "male"

tmp = df[[col, "fraud"]].dropna().copy()
tmp[col] = tmp[col].astype(int)

base = tmp["fraud"].mean()

rate_male = tmp.loc[tmp[col] == 1, "fraud"].mean()
rate_female = tmp.loc[tmp[col] == 0, "fraud"].mean()

pd.DataFrame({
    "p(male=1)": [tmp[col].mean()],
    "fraud_rate_male": [rate_male],
    "fraud_rate_female": [rate_female],
    "lift_male": [rate_male / base],
})

,p(male=1),fraud_rate_male,fraud_rate_female,lift_male
0,0.473788,0.011031,0.010582,1.021903


In [8]:
LABEL = "fraud"

rows_uni = []

model, y, score = fit_logit_and_score(df, [col], y_col=LABEL)

coef = model.params.get(col, np.nan)
pval = model.pvalues.get(col, np.nan)
or_val = float(np.exp(coef)) if pd.notnull(coef) else np.nan

lift = top_decile_lift(y, score)

rows_uni.append({
        "feature": col,
        "coef": float(coef) if pd.notnull(coef) else np.nan,
        "OR": or_val,
        "p_value": float(pval) if pd.notnull(pval) else np.nan,
        "top_decile_rate": lift["top_decile_rate"],
        "top_decile_lift": lift["top_decile_lift"],
        "top_n": lift["top_n"],
    })

uni_df = (
    pd.DataFrame(rows_uni)
      .sort_values("top_decile_lift", ascending=False)
      .reset_index(drop=True)
)

display(uni_df)

,feature,coef,OR,p_value,top_decile_rate,top_decile_lift,top_n
0,male,0.042039,1.042935,0.090381,0.011031,1.021903,288267


> drop!

**years_to_retirement**

In [9]:
col = "years_to_retirement"

s = pd.to_numeric(df[col], errors="coerce")

pd.DataFrame({
    "mean": [s.mean()],
    "std": [s.std()],
    "min": [s.min()],
    "p01": [s.quantile(0.01)],
    "p50": [s.quantile(0.50)],
    "p99": [s.quantile(0.99)],
    "max": [s.max()],
})

,mean,std,min,p01,p50,p99,max
0,14.561422,11.725148,0,0.0,14.0,41.0,48


In [10]:
tmp = df[[col, "fraud"]].dropna().copy()
base = tmp["fraud"].mean()

tmp["bin"] = pd.qcut(tmp[col], 10, duplicates="drop")

g = tmp.groupby("bin")["fraud"]

ret_lift = pd.DataFrame({
    "count": g.size(),
    "fraud_rate": g.mean(),
})

ret_lift["lift"] = ret_lift["fraud_rate"] / base
ret_lift

/tmp/ipykernel_3861508/2165387614.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  g = tmp.groupby("bin")["fraud"]


,count,fraud_rate,lift
bin,,,
"(-0.001, 5.0]",183732,0.012496,1.157616
"(5.0, 10.0]",64872,0.012116,1.122387
"(10.0, 14.0]",59459,0.009452,0.875581
"(14.0, 18.0]",76787,0.009572,0.886701
"(18.0, 21.0]",45794,0.010067,0.932545
"(21.0, 26.0]",65824,0.008994,0.833134
"(26.0, 31.0]",62054,0.010475,0.970334
"(31.0, 48.0]",49908,0.009738,0.902077


In [11]:
LABEL = "fraud"

rows_uni = []

model, y, score = fit_logit_and_score(df, [col], y_col=LABEL)

coef = model.params.get(col, np.nan)
pval = model.pvalues.get(col, np.nan)
or_val = float(np.exp(coef)) if pd.notnull(coef) else np.nan

lift = top_decile_lift(y, score)

rows_uni.append({
        "feature": col,
        "coef": float(coef) if pd.notnull(coef) else np.nan,
        "OR": or_val,
        "p_value": float(pval) if pd.notnull(pval) else np.nan,
        "top_decile_rate": lift["top_decile_rate"],
        "top_decile_lift": lift["top_decile_lift"],
        "top_n": lift["top_n"],
    })

uni_df = (
    pd.DataFrame(rows_uni)
      .sort_values("top_decile_lift", ascending=False)
      .reset_index(drop=True)
)

display(uni_df)

,feature,coef,OR,p_value,top_decile_rate,top_decile_lift,top_n
0,years_to_retirement,-0.009319,0.990724,6.329746e-18,0.012307,1.140109,137640


> drop!